In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pgmpy

df = pd.read_csv(f"credit_risk_dataset.csv")

# df[...] : devuelve solo las filas donde el valor es TRUE. Es un filtro


df_limpio = df.dropna()

print("Filas antes:", len(df))
print("Filas después:", len(df_limpio))
print("Filas eliminadas:", len(df) - len(df_limpio))

# Filtro 1: eliminar las edades imposibles (> 100 años)

df_limpio = df_limpio[df_limpio['person_age'] <= 100]

# Filtro 2: la antiguedad laboral no puede superar (edad - 18)
df_limpio = df_limpio[df_limpio['person_emp_length'] <= df_limpio['person_age'] - 18]



df_limpio = df_limpio.reset_index(drop=True)

columns_names = {
    'person_age': 'Edad de la persona',
    'person_income': 'Ingresos anual ',
    'person_home_ownership': 'Tipo de tenencia de vivienda',
    'person_emp_length': 'Antigüedad laboral ',
    'loan_intent': 'Propósito del préstamo',
    'loan_grade': 'Grado del préstamo',
    'loan_amnt': 'Monto del préstamo',
    'loan_int_rate': 'Tasa de interés del préstamo',
    'loan_status': 'Estado del préstamo',
    'loan_percent_income': 'Porcentaje del ingreso destinado al préstamo',
    'cb_person_default_on_file': 'Default histórico registrado',
    'cb_person_cred_hist_length': 'Duración del historial crediticio',
}

columns_cuantils = ['person_age','person_income','person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']


print(df_limpio[columns_cuantils].describe())


for col in columns_cuantils:
    discrete_columns, edges = pd.qcut(df_limpio[col], q=3, labels=["bajo", "medio", "alto"], retbins=True)

    plt.figure(figsize = (8, 5))
    plt.hist(df_limpio[col], bins = 40, color = 'steelblue', edgecolor = 'black', alpha = 0.5)
    plt.axvline(x = edges[1], color = 'red', linestyle = '--')
    plt.axvline(x = edges[2], color = 'red', linestyle = '--')
    plt.title(columns_names.get(col, col))
    plt.xlabel(columns_names.get(col, col))
    plt.ylabel('Frecuencia')
    print(pd.qcut(df_limpio[col], q=3).value_counts())
    plt.show()

    df_limpio[col + '_disc'] = pd.qcut(df_limpio[col], q=3, labels=["bajo", "medio", "alto"])




# print(df_limpio.isna().sum())

# print(df_limpio[[c+'_disc' for c in columns_cuantils]].nunique())

### Parte 1 - Red Bayesiana

Las variables a ocupar son:

A = person_age: Edad del solicitante  
B = person_income: Ingreso anual  
C = person_home_ownership: Tipo de tenencia de vivienda (arrienda / hipoteca / propia / otro)  
D = person_emp_length: Años de antigüedad laboral  
E = loan_intent: Propósito del préstamo (educación, salud, personal, etc.)  
F = loan_grade: Grado del préstamo según solvencia (A = alta solvencia / bajo riesgo; G = la más baja / mayor riesgo)  
G = loan_amnt: Monto solicitado  
H = loan_int_rate: Tasa de interés del préstamo  
I = loan_status: Estado: (0 = no default, 1 = default)  
J = loan_percent_income: Monto del préstamo como % del ingreso  
K = cb_person_default_on_file: Si tiene un default histórico registrado (Y/N)  
L = cb_person_cred_hist_length: Largo del historial crediticio (años)  


JUSTIFICAR DEPENDENCIAS


A → {E, B, L, D}
B → {J, C, G, H, F, K}
G → J
J → I
E → C, D → C
{F, H, J, K, G} → I

1. Edad: A → {E,B,L,D}

La edad condiciona aspectos asociados a la etapa de vida del solicitante, como ingreso, antigüedad laboral, historial crediticio y propósito del préstamo.

2. Ingreso: B → {J,C,G,H,F,K}

El ingreso representa la capacidad económica del solicitante y se relaciona con el monto solicitado, condiciones del crédito, vivienda y antecedentes crediticios.

3. Tenencia de vivienda: {B,D,E\} → C

La tenencia de vivienda se relaciona con la capacidad económica, estabilidad laboral y necesidades financieras del solicitante.

4. Riesgo de default: {F,H,J,K,G\} → I

El default depende de factores que representan riesgo y capacidad de pago, como grado, tasa, monto, proporción del ingreso y antecedentes de incumplimiento.


Las inferencias más interesantes tienen como objetivo loan_status (I):

1. Influencia de la carga financiera sobre el default: P(I=1|J= alto)

    Determinar cómo cambia la probabilidad de default cuando el monto del préstamo representa un porcentaje alto del ingreso. Esto permite evaluar si una mayor carga financiera relativa aumenta el riesgo de incumplimiento.

2. Influencia del historial de default y grado crediticio: P(I=1| K=Y,F=bajo)
    Calcular la probabilidad de default cuando el solicitante ya posee un default registrado y además presenta un grado crediticio de mayor riesgo. Permite analizar cómo dos indicadores de riesgo combinados modifican la probabilidad de incumplimiento.

3. Perfil de alto riesgo combinando evidencias: P(I=1|J=alto, H=alta, G=alto, K=Y)

    Estimar la probabilidad de default para una persona cuyo préstamo representa una proporción elevada de sus ingresos, posee una tasa de interés alta, solicita un monto elevado y tiene antecedentes de default. Esta inferencia aprovecha mejor la red bayesiana al combinar múltiples evidencias para evaluar un perfil crediticio completo.



Necesitamos que cada variable tenga un número finito y pequeño de categorías.
El problema actual con el dataset es que algunas columnas tienen numeros continuos que podrian tomar infinitos valores
Por ende, se discretizaran: tomar las variables continuas y ingresarlas en una "caja" (bins); que para este caso, cada bin tendra aproximadamente la misma cantidad de gente
Esto sera hecho diviendo los datos en cuantiles:
  - si se divide la variable "person_income" en 4 cajas, cada caja tiene un 25% de los datos.

Estas seran las variables a discretizar(que son numéricas):
person_age
person_income
person_emp_length
loan_amnt
loan_int_rate
loan_percent_income
cb_person_cred_hist_length


Cada caja que se agrega tiene un costo, y el costo cae sobre las CPD's

Mirando el nodo objetivo I (status) Tiene 5 padres: F, H, J, K, G
La tabla de I tiene que dar una probabilidad para cada combinación posible de valores de sus 5 padres.
Si cada padre tiene "n" cajas, el numero de combinaciones es: n * n * n * n * n = n^5


Ahora, porque se elegiria cuantiles para la variable "income"? Porque es una variable muy sesgada y los cuantiles resuelven este problema
Pero para la variable "person_age" que va de 20 a 100, que es una distribucion compacta, no existe el problema que tiene la variable "income".

Entonces, la forma de discretización se elige según la forma de cada distribución =>
 - Variable sesgada: income -> Cuantiles.
 - Variable compacta/simétrica -> rangos fijos funciona bien, ya que tiene una distribución compacta.

Cada variable tiene su necesidad segun su histograma.

A : person_age => cuantiles: media 28.7 > med 27
B : person_income => cuantiles: media 65k > med 54k
D : person_emp_length => cuantiles: media 3.5 > med 3
G : loan_amnt => cuantiles: media 9481 > med 8000
H : loan_int_rate => rangos fijos: mediana ≈ 11
J : loan_percent_income => cuantiles: media 0.17 > med 0.15
L : cb_person_cred_hist_length => cuantiles: media 6.3 > med 5



### Construir el modelo pgmpy

- Declaracion de la red Bayesiana.
    En esta parte, el grafo construido se convierte en el modelo probabilisto real
- Estructuras de la red
    . La estructura (el DAG).
    . Los parámetros (las CPDs): las probabilidades condicionales de cada nodo dado sus padres.

Estimas las CPDs desde los datos es aplicar la máxima verosimilitud

Para aplicar el modelo, se usa el metodo DiscreteBayesianNetwork, y se le da como parametro las dependencias de la red, descritas en df_edges

Para

In [ ]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.parameter_estimator import DiscreteBayesianEstimator


# Descripcion del grafo propuesto:

df_edges = [
    ('person_age_disc', 'loan_intent'),                   # A → E
    ('person_age_disc', 'person_income_disc'),                 # A → B
    ('person_age_disc', 'cb_person_cred_hist_length_disc'),    # A → L
    ('person_age_disc', 'person_emp_length_disc'),             # A → D
    ('loan_intent', 'person_home_ownership'),        # E → C
    ('person_income_disc', 'loan_percent_income_disc'),        # B → J
    ('person_income_disc', 'person_home_ownership'),      # B → C
    ('person_income_disc', 'loan_amnt_disc'),                  # B → G
    ('person_income_disc', 'loan_int_rate_disc'),              # B → H
    ('person_income_disc', 'loan_grade'),                 # B → F
    ('person_income_disc', 'cb_person_default_on_file'),  # B → K
    ('person_emp_length_disc', 'person_home_ownership'),  # D → C
    ('loan_grade', 'loan_status'),                   # F → I
    ('loan_int_rate_disc', 'loan_status'),                # H → I
    ('loan_percent_income_disc', 'loan_status'),          # J → I
    ('cb_person_default_on_file', 'loan_status'),    # K → I
    ('loan_amnt_disc', 'loan_status'),                    # G → I
]

# Movimiento 1 — estructura
model = DiscreteBayesianNetwork(df_edges)

# Movimiento 2 — parámetros
    # solo las columnas que son nodos
data_model = df_limpio[list(model.nodes())]

estimator = DiscreteBayesianEstimator(prior_type='BDeu', equivalent_sample_size=10)
model.fit(data_model, estimator = estimator)

print(model.check_model())

---
## 🤖 Desde aquí: propuesta de estructura de Claude (Parte 1)

Lo de arriba es exactamente lo que tú escribiste, sin ningún cambio. Lo que sigue es **mi sugerencia**
de cómo mapear tu contenido a la estructura de 36 celdas que pide `Especificacion_Celdas_T1.pdf`
(M1–M12 y C1–C9, en el orden exacto del mapa). Todo lo de aquí en adelante empieza con 🤖.

- Las celdas de código que corresponden a algo **que ya escribiste** (C1, C3, C4, C5, C6) traen
  el código **real, copiado tal cual** desde tus celdas 1 y 5 de arriba — puedes ejecutarlas solas.
  Quedan duplicadas respecto a tus celdas originales a propósito, para que compares y elijas una
  sola versión (borra la que no uses).
- Las celdas que son **nuevas** (C2, C7, C8, C9) quedan **comentadas por completo** (cada línea
  empieza con `#`), así no ejecutan nada ni tiran error — son ejemplos de la API, no la solución
  rellenada con tus valores. Rellenarlas es tu parte.

# Tarea 1 — Inteligencia Artificial (CIT-2013)

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**

## [Nombre del ramo y universidad]

- Nombres completos de los dos (o tres) integrantes, en negrita
- Fecha de entrega
- Nombre del ramo y universidad
- Índice breve de las dos partes (Red Bayesiana / Markov y HMM)
- URL de ambos datasets utilizados, con su tamaño (filas × columnas)

## Declaración de uso de herramientas de IA generativa

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**


- Una frase de apertura + lista de propósitos **específicos** (a, b, c) + una frase final de exclusión
- Declarar explícitamente qué **no** se delegó: diseño de la red, selección de variables, criterios de discretización, análisis de resultados
- Declarar solo lo que efectivamente se usó (ni más, ni menos)

In [ ]:
# 🤖 Código real, copiado tal cual desde tu notebook (no modifiqué ninguna línea).
# C1 · Imports — copiados de tu Celda 1 (líneas 1-4) y tu Celda 5 (líneas 1-3).

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pgmpy
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.parameter_estimator import DiscreteBayesianEstimator

In [ ]:
# 🤖 Sugerencia de Claude — celda placeholder, aún no es código tuyo.
# C2 · Semilla y constantes — esto SÍ es nuevo, no está en tu código actual.

# np.random.seed(42)  # o: rng = np.random.default_rng(42)
# DATASET_PATH = "credit_risk_dataset.csv"
# TARGET = "loan_status"


## Parte 1 — Red Bayesiana

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**


### 1.0 Dataset y variables

Versión en tabla de tu glosario A–L (celda 2 de arriba), como sugiere la ficha (columna "Tipo"
derivada de las variables que ya están en `columns_cuantils` en tu código):

| Símbolo | Columna | Descripción | Tipo |
|---|---|---|---|
| A | person_age | Edad del solicitante | Numérica → se discretiza |
| B | person_income | Ingreso anual | Numérica → se discretiza |
| C | person_home_ownership | Tipo de tenencia de vivienda (arrienda / hipoteca / propia / otro) | Categórica |
| D | person_emp_length | Años de antigüedad laboral | Numérica → se discretiza |
| E | loan_intent | Propósito del préstamo (educación, salud, personal, etc.) | Categórica |
| F | loan_grade | Grado del préstamo según solvencia (A = alta solvencia / bajo riesgo; G = la más baja / mayor riesgo) | Categórica |
| G | loan_amnt | Monto solicitado | Numérica → se discretiza |
| H | loan_int_rate | Tasa de interés del préstamo | Numérica → se discretiza |
| I | loan_status | Estado: (0 = no default, 1 = default) | Categórica (**variable objetivo**) |
| J | loan_percent_income | Monto del préstamo como % del ingreso | Numérica → se discretiza |
| K | cb_person_default_on_file | Si tiene un default histórico registrado (Y/N) | Categórica |
| L | cb_person_cred_hist_length | Largo del historial crediticio (años) | Numérica → se discretiza |

Falta agregar: por qué se eligió este dataset, verificación de filas × columnas ≥ 8.000×12,
y por qué la variable objetivo es "relevante".

### 1.1 Limpieza de datos

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**


- Lista numerada de los filtros aplicados: qué hace cada uno, cuántas filas elimina, y el criterio que lo justifica
- Para `dropna`: en qué columnas se concentran los nulos y por qué eliminar en vez de imputar
- Para `emp_length ≤ age − 18`: el supuesto explícito de que nadie trabaja antes de los 18
- Párrafo final **"Efecto sobre la muestra"**: de N filas iniciales a N filas finales, y el cambio en la tasa de default
- Confirmación de que se sigue cumpliendo el mínimo de 8.000 filas

In [ ]:
# 🤖 Código real, copiado tal cual desde tu notebook (no modifiqué ninguna línea).
# C3 · Carga y limpieza — copiado de tu Celda 1, desde `df = pd.read_csv(...)` hasta `reset_index`.

df = pd.read_csv(f"credit_risk_dataset.csv")

# df[...] : devuelve solo las filas donde el valor es TRUE. Es un filtro


df_limpio = df.dropna()

print("Filas antes:", len(df))
print("Filas después:", len(df_limpio))
print("Filas eliminadas:", len(df) - len(df_limpio))

# Filtro 1: eliminar las edades imposibles (> 100 años)

df_limpio = df_limpio[df_limpio['person_age'] <= 100]

# Filtro 2: la antiguedad laboral no puede superar (edad - 18)
df_limpio = df_limpio[df_limpio['person_emp_length'] <= df_limpio['person_age'] - 18]



df_limpio = df_limpio.reset_index(drop=True)

### 1.2 Discretización

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**


Tu celda 3 de arriba ya cubre buena parte de esto (por qué discretizar, por qué cuantiles,
media vs. mediana por variable). Según la ficha, falta:

- Tabla de 5 columnas (Variable · Media · Mediana · Forma · Estrategia)
- El cálculo combinatorio del número de bins considerando que `loan_grade` tiene 7 niveles (no solo 3^5)
- Listado de las categóricas y cuántos niveles tiene cada una
- La decisión sobre `loan_grade`: ¿se dejan los 7 niveles o se agrupan?
- Nota de implementación: las columnas `_disc` se agregan sin sobrescribir las originales

In [ ]:
# 🤖 Código real, copiado tal cual desde tu notebook (no modifiqué ninguna línea).
# C4 · qcut + verificación — copiado de tu Celda 1, desde `columns_names = {...}` hasta el final del for.

columns_names = {
    'person_age': 'Edad de la persona',
    'person_income': 'Ingresos anual ',
    'person_home_ownership': 'Tipo de tenencia de vivienda',
    'person_emp_length': 'Antigüedad laboral ',
    'loan_intent': 'Propósito del préstamo',
    'loan_grade': 'Grado del préstamo',
    'loan_amnt': 'Monto del préstamo',
    'loan_int_rate': 'Tasa de interés del préstamo',
    'loan_status': 'Estado del préstamo',
    'loan_percent_income': 'Porcentaje del ingreso destinado al préstamo',
    'cb_person_default_on_file': 'Default histórico registrado',
    'cb_person_cred_hist_length': 'Duración del historial crediticio',
}

columns_cuantils = ['person_age','person_income','person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']


print(df_limpio[columns_cuantils].describe())


for col in columns_cuantils:
    discrete_columns, edges = pd.qcut(df_limpio[col], q=3, labels=["bajo", "medio", "alto"], retbins=True)

    plt.figure(figsize = (8, 5))
    plt.hist(df_limpio[col], bins = 40, color = 'steelblue', edgecolor = 'black', alpha = 0.5)
    plt.axvline(x = edges[1], color = 'red', linestyle = '--')
    plt.axvline(x = edges[2], color = 'red', linestyle = '--')
    plt.title(columns_names.get(col, col))
    plt.xlabel(columns_names.get(col, col))
    plt.ylabel('Frecuencia')
    print(pd.qcut(df_limpio[col], q=3).value_counts())
    plt.show()

    df_limpio[col + '_disc'] = pd.qcut(df_limpio[col], q=3, labels=["bajo", "medio", "alto"])




# print(df_limpio.isna().sum())

# print(df_limpio[[c+'_disc' for c in columns_cuantils]].nunique())

### 1.3 Estructura propuesta

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**


#### Justificación de las dependencias

Tu celda 2 de arriba ya tiene el resumen de aristas y 4 párrafos de justificación agrupados por bloque.
La ficha pide además:

- Frase de encabezado: 12 nodos, 17 aristas, variable objetivo con 5 padres
- Idealmente una tabla de 2 columnas (Arista · Mecanismo), una fila por arista
- Tratamiento especial de **G → J**: `loan_percent_income` es `loan_amnt / person_income`, una relación funcional y no probabilística

#### Justificación de las independencias

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**


- Frase de apertura: cada arista ausente es una afirmación de independencia condicional, no una omisión neutral
- **Pares mediados:** 2–3 pares de variables que parecen relacionadas pero se dejaron sin arista directa
- **Los tres patrones en la red:** un ejemplo de cadena, uno de bifurcación, uno de colisionador (candidato: `loan_status` con sus 5 padres)
- **Verificación:** mención de que se comprobó con `local_independencies()`

In [ ]:
# 🤖 Código real, copiado tal cual desde tu notebook (no modifiqué ninguna línea).
# C5 · Declarar DAG — copiado de tu Celda 5, desde `df_edges = [...]` hasta `model = DiscreteBayesianNetwork(df_edges)`. Falta la visualización (ver TODO comentado abajo).

# Descripcion del grafo propuesto:

df_edges = [
    ('person_age_disc', 'loan_intent'),                   # A → E
    ('person_age_disc', 'person_income_disc'),                 # A → B
    ('person_age_disc', 'cb_person_cred_hist_length_disc'),    # A → L
    ('person_age_disc', 'person_emp_length_disc'),             # A → D
    ('loan_intent', 'person_home_ownership'),        # E → C
    ('person_income_disc', 'loan_percent_income_disc'),        # B → J
    ('person_income_disc', 'person_home_ownership'),      # B → C
    ('person_income_disc', 'loan_amnt_disc'),                  # B → G
    ('person_income_disc', 'loan_int_rate_disc'),              # B → H
    ('person_income_disc', 'loan_grade'),                 # B → F
    ('person_income_disc', 'cb_person_default_on_file'),  # B → K
    ('person_emp_length_disc', 'person_home_ownership'),  # D → C
    ('loan_grade', 'loan_status'),                   # F → I
    ('loan_int_rate_disc', 'loan_status'),                # H → I
    ('loan_percent_income_disc', 'loan_status'),          # J → I
    ('cb_person_default_on_file', 'loan_status'),    # K → I
    ('loan_amnt_disc', 'loan_status'),                    # G → I
]

# Movimiento 1 — estructura
model = DiscreteBayesianNetwork(df_edges)

# TODO (nuevo): visualizar el grafo, por ejemplo:
# import networkx as nx
# nx.draw(model, with_labels=True, node_color='lightblue', arrows=True)
# print(model.local_independencies("loan_status"))
# print(model.get_independencies())

### 1.4 Estimación de parámetros

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**


Tu celda 4 de arriba ("Construir el modelo pgmpy") ya cubre la idea de MLE y estructura vs. parámetros.
Falta declarar explícitamente:

- Si se usa MLE puro o el estimador bayesiano (ya usas BDeu con `equivalent_sample_size=10` en el código: falta justificar ese número)
- Por qué el riesgo de celdas con cero es real dado el número de combinaciones de padres

In [ ]:
# 🤖 Código real, copiado tal cual desde tu notebook (no modifiqué ninguna línea).
# C6 · fit + check_model — copiado de tu Celda 5, desde `data_model = ...` hasta `print(model.check_model())`.

# Movimiento 2 — parámetros
    # solo las columnas que son nodos
data_model = df_limpio[list(model.nodes())]

estimator = DiscreteBayesianEstimator(prior_type='BDeu', equivalent_sample_size=10)
model.fit(data_model, estimator = estimator)

print(model.check_model())

### 1.5 Inferencias sobre la red

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**


Tus 3 inferencias ya están planteadas en la celda 2 de arriba. Falta agregar:

- Frase de apertura: qué es inferir en una red bayesiana
- Etiquetar cada consulta como predictiva o diagnóstica (las tres actuales son predictivas — considera cambiar una por una diagnóstica)

In [ ]:
# 🤖 Sugerencia de Claude — celda placeholder, aún no es código tuyo.
# C7 · Ejecutar las 3 consultas — nuevo, no está en tu código actual.

# from pgmpy.inference import VariableElimination
# infer = VariableElimination(model)

# q1 = infer.query(variables=["loan_status"], evidence={"loan_percent_income_disc": "alto"})
# q2 = infer.query(variables=["loan_status"], evidence={"cb_person_default_on_file": "Y",
#                                                        "loan_grade": "bajo"})
# q3 = infer.query(variables=["loan_status"], evidence={"loan_percent_income_disc": "alto",
#                                                        "loan_int_rate_disc": "alto",
#                                                        "loan_amnt_disc": "alto",
#                                                        "cb_person_default_on_file": "Y"})
# print(q1); print(q2); print(q3)

# Los valores exactos de evidence deben coincidir con las etiquetas reales de tus columnas *_disc
# (revisa con df_limpio['loan_percent_income_disc'].unique(), etc. antes de escribirlos).

#### Interpretación de los resultados

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**


Por cada inferencia, 4 viñetas fijas:
- **Resultado:** el valor numérico
- **Qué significa en el dominio:** traducción a lenguaje crediticio
- **Comparación con la base:** contraste contra P(default) sin evidencia
- **Qué parte de la estructura lo produce**

### 1.6 Estructura aprendida automáticamente

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**


- El algoritmo elegido (p.ej. HillClimbSearch + BicScore, o PC), con precisión
- A qué familia pertenece y qué optimiza
- Las dos limitaciones: óptimo local, y equivalencia de Markov

In [ ]:
# 🤖 Sugerencia de Claude — celda placeholder, aún no es código tuyo.
# C8 · Aprender la segunda estructura y re-ajustar — nuevo.

# from pgmpy.estimators import HillClimbSearch, BicScore

# hc = HillClimbSearch(data_model)
# dag_aprendido = hc.estimate(scoring_method=BicScore(data_model))
# print("Aristas aprendidas:", sorted(dag_aprendido.edges()))

# model_aprendido = DiscreteBayesianNetwork(dag_aprendido.edges())
# model_aprendido.fit(data_model, estimator=BayesianEstimator,
#                      prior_type='BDeu', equivalent_sample_size=10)
# print(model_aprendido.check_model())

In [ ]:
# 🤖 Sugerencia de Claude — celda placeholder, aún no es código tuyo.
# C9 · Métricas de comparación — nuevo.

# from pgmpy.estimators import BicScore
# score = BicScore(data_model)
# print("BIC propuesta:", score.score(model))
# print("BIC aprendida:", score.score(model_aprendido))

# a, b = set(model.edges()), set(model_aprendido.edges())
# print("solo en la propuesta:", a - b)
# print("solo en la aprendida:", b - a)
# print("en ambas:", a & b)

### 1.7 Comparación: red propuesta vs. red aprendida

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**


#### Eje 1 — Estructura
#### Eje 2 — Ajuste a los datos
#### Eje 3 — Inferencias
#### Conclusión